## Step 1: Define the calculator tool

The simplest tool — takes a math expression as a string, evaluates it safely,
and returns the result. This is what the agent will call whenever a question
requires exact arithmetic, which LLMs are notoriously unreliable at doing
directly.

In [1]:
import ast
import operator

# safe evaluation - only allow basic math operations, not arbitrary code execution
ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    elif isinstance(node, ast.BinOp):
        return ALLOWED_OPS[type(node.op)](safe_eval(node.left), safe_eval(node.right))
    elif isinstance(node, ast.UnaryOp):
        return ALLOWED_OPS[type(node.op)](safe_eval(node.operand))
    else:
        raise ValueError(f"Unsupported expression: {node}")

def calculator(expression: str) -> str:
    try:
        tree = ast.parse(expression, mode='eval')
        result = safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# test it
print(calculator("47 * 892"))
print(calculator("(15 + 3) / 2"))
print(calculator("2 ** 10"))

41924
9.0
1024


## Step 2: Define the web search tool

For questions requiring current information the LLM doesn't know, the agent
needs to search the web. We'll use a simple search API wrapper.

In [3]:
!pip install ddgs -q

from ddgs import DDGS

def web_search(query: str, max_results: int = 3) -> str:
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        formatted = "\n\n".join([f"{r['title']}: {r['body']}" for r in results])
        return formatted
    except Exception as e:
        return f"Error: {e}"

# test it
print(web_search("current population of Nepal"))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.3 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Demographics of Nepal - Wikipedia: The population of Nepal has been steadily rising in recent decades. In the June 2001 census, there was a population of about 23 million in Nepal. [9] The population increased by 5 million from the preceding 1991 census; the growth rate is 2.3%. [9] The current population is roughly 30 million which contributes to an increase of about 3 million people every 5 years. Sixty caste and linguistic ...

Nepal Population (2026) - Worldometer: Population of Nepal: current, historical, and projected population, growth rate, immigration, median age, total fertility rate (TFR), population density, urbanization ...

Nepal Population 2026: Nepal represents 0.37% of the total global population and 0.59% of Asia's population. Nepal has a population density of 206.7 people per km² (approximately 535.3 per mi²), making it the 71st most densely populated country in the world. Currently, 30.3% of the population resides in urban areas, a figure that is increasing annually

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Step 3: Define the code execution tool

For tasks requiring actual computation beyond simple arithmetic (e.g., data
processing, string manipulation, generating a list), the agent needs to
run real Python code. We use a restricted execution environment to avoid
security risks from arbitrary code.

In [4]:
import io
import contextlib

def execute_code(code: str) -> str:
    output_buffer = io.StringIO()
    try:
        # restricted globals - no access to file system, imports, etc.
        safe_globals = {"__builtins__": {"print": print, "range": range, "len": len,
                                          "sum": sum, "min": min, "max": max,
                                          "sorted": sorted, "list": list, "dict": dict,
                                          "str": str, "int": int, "float": float}}
        with contextlib.redirect_stdout(output_buffer):
            exec(code, safe_globals)
        result = output_buffer.getvalue()
        return result if result else "Code executed successfully (no output printed)"
    except Exception as e:
        return f"Error: {e}"

# test it
print(execute_code("print(sum([1, 2, 3, 4, 5]))"))
print(execute_code("x = [i**2 for i in range(10)]\nprint(x)"))

15

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

## Step 4: Build the agent loop (ReAct pattern)

Now we wire the three tools together with an LLM that decides which tool
to call, in a loop: the LLM outputs a Thought and an Action, we execute
the action and feed back an Observation, and repeat until the LLM decides
it has enough information to give a Final Answer.

We instruct the LLM (via prompt) to always respond in a structured format
so we can parse out its intended action.

In [6]:
import re
from google.genai import types

TOOLS_DESCRIPTION = """
You have access to these tools:
1. calculator(expression) - evaluates a math expression, e.g. calculator("47 * 892")
2. web_search(query) - searches the web, e.g. web_search("population of Nepal")
3. execute_code(code) - runs Python code and returns printed output, e.g. execute_code("print(sum([1,2,3]))")

Respond in this exact format:
Thought: <your reasoning about what to do next>
Action: <tool_name>(<input>)

Once you have enough information to answer, respond instead with:
Thought: <your reasoning>
Final Answer: <your answer to the user>
"""

def run_tool(action_text):
    match = re.match(r'(\w+)\((.*)\)', action_text.strip(), re.DOTALL)
    if not match:
        return "Error: could not parse action"
    tool_name, tool_input = match.group(1), match.group(2).strip().strip('"').strip("'")

    if tool_name == "calculator":
        return calculator(tool_input)
    elif tool_name == "web_search":
        return web_search(tool_input)
    elif tool_name == "execute_code":
        return execute_code(tool_input)
    else:
        return f"Error: unknown tool {tool_name}"

def run_agent(question, gemini_client, max_steps=5):
    history = f"{TOOLS_DESCRIPTION}\n\nQuestion: {question}\n"

    for step in range(max_steps):
        response = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=history
        )
        text = response.text
        print(f"--- Step {step+1} ---")
        print(text)

        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()

        action_match = re.search(r'Action:\s*(.+)', text)
        if action_match:
            action_text = action_match.group(1).strip()
            observation = run_tool(action_text)
            print(f"Observation: {observation}\n")
            history += f"{text}\nObservation: {observation}\n"
        else:
            return "Agent did not produce a valid action or final answer."

    return "Max steps reached without a final answer."

# test it
from google import genai
from google.colab import userdata
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

answer = run_agent("What is 15% of the current population of Nepal?", client)
print("\n=== FINAL ANSWER ===")
print(answer)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

--- Step 1 ---
Thought: I need to find the current population of Nepal first.
Action: web_search("current population of Nepal 2024")


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Observation: Nepal Population (2026) - Worldometer: The current population of Nepal is 29,582,979 as of Thursday, September 10, 2026, based on Worldometer's elaboration of the latest United Nations data 1. Nepal 2026 population is estimated at 29,629,410 people at mid-year. Nepal population is equivalent to 0.36% of the total world population.

Nepal Demographics 2024 (Population, Age, Sex, Trends ...: The 2024 population density in Nepal is 207 people per Km 2 (536 people per mi 2), calculated on a total land area of 143,350 Km2 (55,348 sq. miles). A Population pyramid (also called "Age-Sex Pyramid") is a graphical representation of the age and sex of a population. Types:

Nepal Population & Demographics (2026), Population Review: Nepal has a population of 29,694,614, ranking # 134 in the world. The capital is Kathmandu. Located in South Asia. The country of Nepal had a population of 10,123,658 in 1960 and 29,651,054 in 2024, a 2.9× growth over 64 years. The peak was 29,715,436 in 202

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 31.273212208s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag